# Example 7.9 / 1: Mark a Lazy Row and a User Cut

Companion notebook for **7.9 Lazy Constraints and User Cuts** in the AMPL lab Documentation, section "Mark a Lazy Row and a User Cut". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Mark a Lazy Row and a User Cut

MP drivers that support the convention read positive `.lazy` values as lazy constraints and
negative values as user cuts; the option `mip:lazy` (default 3, both) controls which are
recognized. In the knapsack below, `Conflict` is a business rule (items `i2` and `i6` cannot both
be packed), so it is part of the model. `Cover` is a cover inequality: items `i1`, `i3` and `i7`
weigh 69 together, more than the capacity 63, so at most two of them fit, and every integer
solution already satisfies it.

First check the roles with HiGHS, which ignores the suffix: dropping a valid cut must not change
the optimum, while dropping a required row may. amplpy's `drop()` and `restore()` remove a
constraint from the next solves and bring it back:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
suffix lazy IN;            # 1 marks a lazy constraint, -1 a user cut
set ITEMS;                 # items that can be packed
param value {ITEMS} > 0;   # value of each item
param weight {ITEMS} > 0;  # weight of each item
param volume {ITEMS} > 0;  # volume of each item
param max_weight > 0;      # weight capacity
param max_volume > 0;      # volume capacity
var Take {ITEMS} binary;   # 1 if the item is packed
maximize Total_Value: sum {i in ITEMS} value[i] * Take[i];  # total value packed
subject to Weight: sum {i in ITEMS} weight[i] * Take[i] <= max_weight;  # weight capacity
subject to Volume: sum {i in ITEMS} volume[i] * Take[i] <= max_volume;  # volume capacity
subject to Conflict: Take["i2"] + Take["i6"] <= 1;           # business rule: required
subject to Cover: Take["i1"] + Take["i3"] + Take["i7"] <= 2;  # implied by Weight: a valid cut

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
items = ["i1", "i2", "i3", "i4", "i5", "i6", "i7", "i8"]
ampl.set["ITEMS"] = items
ampl.param["value"] = dict(zip(items, [25, 47, 44, 18, 33, 48, 40, 50]))
ampl.param["weight"] = dict(zip(items, [23, 7, 24, 5, 20, 13, 22, 12]))
ampl.param["volume"] = dict(zip(items, [11, 27, 20, 22, 22, 20, 17, 25]))
ampl.param["max_weight"] = 63
ampl.param["max_volume"] = 82

In [5]:
# run.py
def solve(label):
    ampl.solve(solver="highs", verbose=False)
    taken = [i for i, v in ampl.var["Take"].to_dict().items() if v > 0.5]
    print(label, "|", ampl.solve_result, ampl.obj["Total_Value"].value(), taken)

solve("all rows")
for name in ["Cover", "Conflict"]:
    ampl.con[name].drop()
    solve("without " + name)
    ampl.con[name].restore()

all rows | solved 142.0 ['i3', 'i6', 'i8']
without Cover | solved 142.0 ['i3', 'i6', 'i8']
without Conflict | solved 153.0 ['i1', 'i2', 'i5', 'i6']


### Pure AMPL: data.dat and model.run

Without `Cover` the optimum is unchanged, as a valid cut requires. Without `Conflict` the solver
packs `i2` and `i6` together and earns more: that row is part of the intended model and may only
be marked lazy, never as a user cut.

The same check in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param: ITEMS: value weight volume :=
  i1  25  23  11
  i2  47   7  27
  i3  44  24  20
  i4  18   5  22
  i5  33  20  22
  i6  48  13  20
  i7  40  22  17
  i8  50  12  25 ;
param max_weight := 63;
param max_volume := 82;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
solve;
display Total_Value;
drop Cover;
solve;
display Total_Value;
restore Cover;
drop Conflict;
solve;
display Total_Value;

Total_Value = 142

Total_Value = 142

Total_Value = 153



### Another way in amplpy

Now pass the annotations to Gurobi. AMPL's own presolve is turned off so that the two rows reach
the solver as written, and `let` sets the suffix values; with the same `model.mod` and
`data.py`:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
items = ["i1", "i2", "i3", "i4", "i5", "i6", "i7", "i8"]
ampl.set["ITEMS"] = items
ampl.param["value"] = dict(zip(items, [25, 47, 44, 18, 33, 48, 40, 50]))
ampl.param["weight"] = dict(zip(items, [23, 7, 24, 5, 20, 13, 22, 12]))
ampl.param["volume"] = dict(zip(items, [11, 27, 20, 22, 22, 20, 17, 25]))
ampl.param["max_weight"] = 63
ampl.param["max_volume"] = 82

In [11]:
# run.py
ampl.option["presolve"] = 0
ampl.eval("let Conflict.lazy := 1; let Cover.lazy := -1;")
ampl.solve(solver="gurobi", verbose=False)
taken = [i for i, v in ampl.var["Take"].to_dict().items() if v > 0.5]
print(ampl.solve_result, ampl.obj["Total_Value"].value(), taken)

solved 142.0 ['i3', 'i6', 'i8']
